# Модуль 2. Урок 6: Продуктовые паттерны на SQL (Когорты & Retention)

Расчет даты первого заказа для каждого пользователя (Когорта) и распределение повторных покупок.


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r"d:\vibe_app\main_data_analytics\course\datasets\ecommerce.db")

query = """
WITH first_orders AS (
    SELECT 
        customer_id,
        MIN(SUBSTR(order_date, 1, 7)) AS cohort_month
    FROM orders
    WHERE status = 'Completed'
    GROUP BY customer_id
),
customer_activities AS (
    SELECT 
        o.customer_id,
        fo.cohort_month,
        SUBSTR(o.order_date, 1, 7) AS order_month
    FROM orders o
    JOIN first_orders fo ON o.customer_id = fo.customer_id
    WHERE o.status = 'Completed'
)
SELECT 
    cohort_month,
    COUNT(DISTINCT customer_id) AS total_cohort_users,
    COUNT(DISTINCT CASE WHEN order_month = cohort_month THEN customer_id END) AS month_0,
    COUNT(DISTINCT CASE WHEN order_month > cohort_month THEN customer_id END) AS repeat_buyers
FROM customer_activities
GROUP BY cohort_month
ORDER BY cohort_month
LIMIT 6;
"""
print(pd.read_sql(query, conn))
